# Loan Shark Escape — Hackathon demo

**One line:** a borrower must sequence monthly actions to escape predatory debt before **spiral lock** — when cash falls below the sum of rollover minimums, escape is mathematically impossible.

This notebook runs **two policies** on `lse-medium`: a naive always–pay-minimum agent (hits spiral lock) vs a scripted “smart” agent that uses the **credit union window** and **NGO** milestones, then pays loans off.

## Prerequisite

Start the API server (from repo root):

```bash
uvicorn server.app:app --host 0.0.0.0 --port 7860
```

Then run the cells below.

In [ ]:
import asyncio
import json
import os
from typing import Any, Callable

import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

from client import LoanSharkEscapeEnv

API_BASE_URL = os.environ.get("API_BASE_URL", "http://localhost:7860")
TASK_ID = "lse-medium"

In [ ]:
async def run_episode(policy: Callable[[dict[str, Any]], int], label: str) -> dict[str, Any]:
    async with LoanSharkEscapeEnv(API_BASE_URL) as env:
        obs = await env.reset(TASK_ID)
        done = False
        steps = 0
        reward_sum = 0.0
        fee_trace: list[float] = []

        while not done and steps < 60:
            action = int(policy(obs))
            payload = await env.step({"action": action})
            reward_sum += float(payload.get("reward", 0.0))
            done = bool(payload.get("done", False))
            obs = payload.get("observation", payload)
            fee_trace.append(float(obs.get("total_fees_paid", 0.0)))
            steps += 1

        grade = await env.evaluate()
        state = await env.state()

    return {
        "agent": label,
        "months": steps,
        "total_reward": round(reward_sum, 2),
        "fees": round(float(state.get("total_fees_paid", 0.0)), 2),
        "spiral_lock": bool(state.get("spiral_lock", False)),
        "escaped": bool(state.get("all_loans_cleared", False)),
        "grader_reward": grade.get("reward", 0.0),
        "grader": grade,
        "fee_trace": fee_trace,
    }


def naive_policy(_obs: dict[str, Any]) -> int:
    """Always pay minimum rollover (the behavioral trap)."""
    return 1


def smart_scripted_policy(obs: dict[str, Any]) -> int:
    """Use credit union once when the window opens, NGO on configured months, then clear balances."""
    routes = obs.get("escape_routes", {})
    month = int(obs.get("month", 0))
    cash = float(obs.get("cash_on_hand", 0.0))
    loans = obs.get("loans", [])
    active = [L for L in loans if float(L.get("balance", 0)) > 0]
    if not active:
        return 4
    if routes.get("credit_union_available", False) and not obs.get("credit_union_used", False):
        return 2
    if routes.get("ngo_help_available", False) and month in (6, 12):
        return 3
    total_bal = sum(float(L["balance"]) for L in active)
    if cash >= total_bal - 1e-6:
        return 0
    return 1

In [ ]:
naive = asyncio.run(run_episode(naive_policy, "Naive (always pay minimum)"))
smart = asyncio.run(run_episode(smart_scripted_policy, "Smart (escape windows + payoff)"))

compare = pd.DataFrame([naive, smart])[
    ["agent", "months", "fees", "spiral_lock", "escaped", "grader_reward"]
]
display(compare)

naive_fees = float(naive["fees"])
smart_fees = float(smart["fees"])
reduction = ((naive_fees - smart_fees) / naive_fees * 100) if naive_fees > 0 else 0.0
print(f"Fee reduction (smart vs naive): {reduction:.1f}%")
print("\nNaive grader breakdown:", json.dumps(naive["grader"]["tests"], indent=2))
print("Smart grader breakdown:", json.dumps(smart["grader"]["tests"], indent=2))

In [ ]:
if plt is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    agents = [naive["agent"], smart["agent"]]
    fees = [naive["fees"], smart["fees"]]
    colors = ["#c0392b", "#27ae60"]
    axes[0].bar(agents, fees, color=colors)
    axes[0].set_title("Total fees paid (lower is better)")
    axes[0].set_ylabel("Fees")
    axes[0].tick_params(axis="x", rotation=15)

    axes[1].plot(naive["fee_trace"], label="Naive", color="#c0392b", linewidth=2)
    axes[1].plot(smart["fee_trace"], label="Smart", color="#27ae60", linewidth=2)
    axes[1].set_title("Fee burn over time")
    axes[1].set_xlabel("Month (step index)")
    axes[1].set_ylabel("Cumulative fees")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("Install matplotlib to render comparison charts: pip install matplotlib")

## Optional: LLM agent (same API)

Requires `OPENAI_API_KEY` and `pip install openai`. Mirrors `inference.py` — reads the observation JSON and returns action `0`–`4`.

In [ ]:
async def run_llm_episode() -> dict[str, Any]:
    import re
    from openai import OpenAI

    def parse_action(text: str) -> int:
        m = re.search(r"\b([0-4])\b", text or "")
        return int(m.group(1)) if m else 1

    client = OpenAI()
    model = os.environ.get("MODEL_NAME", "gpt-4o-mini")

    async with LoanSharkEscapeEnv(API_BASE_URL) as env:
        obs = await env.reset(TASK_ID)
        done = False
        steps = 0
        while not done and steps < 60:
            prompt = (
                "You are a borrower in a debt-trap simulation. Output exactly one digit 0-4.\n"
                "0 pay off what you can (full priority), 1 minimum rollover only, 2 credit union if available, "
                "3 NGO help if available, 4 wait.\n"
                "Avoid spiral lock: never let cash_on_hand stay below the sum of weekly_fee on active loans.\n"
                f"Observation JSON:\n{json.dumps(obs, indent=2)}"
            )
            out = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=16,
                temperature=0,
            )
            action = parse_action((out.choices[0].message.content or "").strip())
            payload = await env.step({"action": action})
            done = bool(payload.get("done", False))
            obs = payload.get("observation", obs)
            steps += 1
        grade = await env.evaluate()
        state = await env.state()

    return {
        "months": steps,
        "fees": round(float(state.get("total_fees_paid", 0.0)), 2),
        "spiral_lock": state.get("spiral_lock"),
        "escaped": state.get("all_loans_cleared"),
        "grader": grade,
    }


if os.environ.get("OPENAI_API_KEY"):
    llm_result = asyncio.run(run_llm_episode())
    print(json.dumps(llm_result, indent=2, default=str))
else:
    print("Skip: set OPENAI_API_KEY to run the LLM episode.")